# Deploying a trained model

_PyTorch (a complete course)_

**Take the trained weights out of the notebook: export to a Python-free runtime (TorchScript / ONNX), quantize to int8, and serve it behind an API.**

A trained model in a notebook is the start of the work, not the end. The notebook gives you weights that work; production demands a graph that is fast, portable, observable, and cheap to run.

       The core move is export: freeze the model into a standalone graph that no longer needs Python or the training code. TorchScript freezes it into a PyTorch-native serialized graph; ONNX freezes it into a framework-neutral file that other engines can run. Either way, the result is a single artifact you can drop onto a server, a phone, or a C++ binary.

---

This notebook is a practice scaffold for the **Deploying a trained model** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q onnx onnxruntime
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np, os

torch.manual_seed(0)

# ---- a tiny image classifier (3x32x32 -> 10 classes) ----
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 8 * 8, 128)
        self.fc2   = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # 16x16
        x = self.pool(torch.relu(self.conv2(x)))   #  8x8
        x = x.flatten(1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)                          # logits

model = Net().eval()                                # <-- inference mode (the famous must-do)
n_params = sum(p.numel() for p in model.parameters())
print("params:", n_params, "| fp32 MB:", round(n_params * 4 / 1e6, 3))

x = torch.randn(1, 3, 32, 32)                       # one example input
with torch.inference_mode():                        # no autograd graph at serving
    ref = model(x)

# ================================================================
# 1) TORCHSCRIPT: trace (straight-line) and script (keeps control flow)
# ================================================================
ts_traced = torch.jit.trace(model, x)               # records ops for THIS input
ts_scripted = torch.jit.script(model)               # compiles the Python -> graph
ts_traced.save("model_ts.pt")
reloaded = torch.jit.load("model_ts.pt")            # runs WITHOUT the Net class / Python
with torch.inference_mode():
    print("TorchScript matches PyTorch:",
          torch.allclose(reloaded(x), ref, atol=1e-5))

# ================================================================
# 2) ONNX (Open Neural Network Exchange): export + run with onnxruntime
# ================================================================
# (the notebook setup cell installs onnx + onnxruntime)
import onnx, onnxruntime as ort
torch.onnx.export(
    model, x, "model.onnx",
    input_names=["input"], output_names=["logits"],
    opset_version=17,                               # PIN the opset across the boundary
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},  # variable batch size
)
onnx.checker.check_model(onnx.load("model.onnx"))   # structural validity
sess = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
onnx_out = sess.run(["logits"], {"input": x.numpy()})[0]
print("ONNX Runtime matches PyTorch:",
      np.allclose(onnx_out, ref.numpy(), atol=1e-4))   # ALWAYS verify across the boundary

# ================================================================
# 3) DYNAMIC QUANTIZATION: int8 the Linear layers -> smaller + faster
# ================================================================
qmodel = torch.ao.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
).eval()
torch.save(model.state_dict(),  "fp32.pt")
torch.save(qmodel.state_dict(), "int8.pt")
print("fp32 file KB:", round(os.path.getsize("fp32.pt") / 1e3, 1),
      "| int8 file KB:", round(os.path.getsize("int8.pt") / 1e3, 1))
# NOTE: always re-validate accuracy on a held-out set before shipping int8!

# ================================================================
# 4) SERVE: a minimal FastAPI /predict sketch (preprocess shared w/ training)
# ================================================================
SERVE = '''
from fastapi import FastAPI
import torch
app = FastAPI()
model = torch.jit.load("model_ts.pt").eval()      # load the exported graph, eval mode

def preprocess(payload):                           # SAME steps as training -> no skew
    t = torch.tensor(payload["pixels"]).float().reshape(1, 3, 32, 32)
    return (t - 0.5) / 0.5                          # match the training normalization

@app.post("/predict")
def predict(payload: dict):
    x = preprocess(payload)
    with torch.inference_mode():                   # no grad graph per request
        logits = model(x)
    return {"class": int(logits.argmax(1).item())}
'''
print(SERVE)

## Visualize the data & results

_How much smaller and faster does the model get as you move from FP32 eager PyTorch to TorchScript, to ONNX Runtime, to int8 quantization?_

In [ ]:
import numpy as np

# ---- model param count, layer by layer (real architecture from the CODE cell) ----
def conv(cin, cout, k):  return cout * cin * k * k + cout
def lin(fin, fout):      return fin * fout + fout

params = (
    conv(3, 16, 3)          # conv1 -> 448
  + conv(16, 32, 3)         # conv2 -> 4640
  + lin(32 * 8 * 8, 128)    # fc1   -> 262272
  + lin(128, 10)            # fc2   -> 1290
)
print("total params:", params)                      # 268650

# ---- size from param count: float32 = 4 bytes, int8 = 1 byte ----
fp32_mb = params * 4 / 1e6
int8_mb = params * 1 / 1e6
sizes = {
    "FP32 (.pt)":     round(fp32_mb, 3),            # 1.075
    "TorchScript":    round(fp32_mb * 1.01, 3),     # ~same weights + tiny graph overhead
    "ONNX Runtime":   round(fp32_mb, 3),            # same float32 weights
    "int8 quantized": round(int8_mb, 3),            # ~4x smaller -> 0.269
}
print("size MB:", sizes)
print("size reduction (int8):", round(fp32_mb / int8_mb, 2), "x")  # 4.0x

# ---- latency: illustrative, monotonically improving across optimized runtimes ----
latency_ms = {
    "FP32 eager":     12.0,
    "TorchScript":     9.0,
    "ONNX Runtime":    6.5,
    "int8 quantized":  4.0,
}
print("latency ms:", latency_ms)
print("speedup (int8 vs eager):", round(12.0 / 4.0, 2), "x")        # 3.0x

# ---- charts ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
cols = ["#ff7b72", "#4ea1ff", "#c89bff", "#7ee787"]
ax[0].bar(list(sizes.keys()), list(sizes.values()), color=cols)
ax[0].set_ylabel("size (MB)"); ax[0].set_title("model size on disk")
ax[0].tick_params(axis="x", rotation=20)
ax[1].bar(list(latency_ms.keys()), list(latency_ms.values()), color=cols)
ax[1].set_ylabel("latency (ms)"); ax[1].set_title("inference latency (lower is better)")
ax[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your model has an if x.sum() > 0: branch inside forward(). You export it with torch.jit.trace and deploy. Some inputs now give clearly wrong answers. What happened, and how do you fix the export?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what trace records. — _trace runs the model on one example and records only the operations that actually ran — the branch not taken on that example is missing from the graph._
- Realize the branch was frozen. — _Whichever side of the if the example triggered is baked in for every future input, so inputs that should take the other branch are handled wrongly._
- Re-export with torch.jit.script(model). — _script compiles the actual Python control flow, so the if survives as a real branch in the graph._

**Answer:** trace dropped the data-dependent branch (it only saw one path). Use ts = torch.jit.script(model) instead, which preserves the if. Reserve trace for straight-line tensor code.

</details>

**Problem 2.** You quantized your model to int8 to make it 4x smaller and faster. How do you decide whether it is safe to ship?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remember quantization is lossy. — _int8 rounds every weight, so the model's outputs shift slightly; the effect on accuracy varies by model and cannot be assumed._
- Evaluate the quantized model on a held-out validation set. — _The only way to know the real cost is to measure the same metric you used in training on data the model never saw._
- Compare against your accuracy budget. — _Ship only if the drop is within the budget the product can tolerate; otherwise keep float32 or try a less aggressive scheme._

**Answer:** Validate the int8 model on held-out data and compare its accuracy to the float32 model against an explicit budget. Never assume "little loss" — measure it.

</details>

**Problem 3.** Your FastAPI /predict endpoint works but is slow and occasionally returns inconsistent answers for the same input. List the two serving-time fixes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set the model to inference mode at load: model.eval(). — _In training mode dropout randomly zeroes units and batch-norm uses batch stats, so the same input can give different (and wrong) answers._
- Wrap every prediction in with torch.inference_mode():. — _Stops PyTorch from building the autograd graph you never use at serving, cutting per-request memory and latency._

**Answer:** Call model.eval() once at load (fixes the non-determinism from dropout/batch-norm) and run each request inside torch.inference_mode() (fixes the wasted memory/latency). Batching requests further raises throughput.

</details>